In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 깃허브에 올라가는 순간 보안사고
key = os.getenv('OPEN_API_KEY', '')
# == os.environ.get('OPEN_API_KEY', '')

# os.environ['OPEN_API_KEY']

In [ ]:
# decoding
from urllib.parse import unquote, quote

quote(OPEN_API_KEY)
# QHVhAahAshrNunAtQRenvZkOaIta0uRo9cVcSp%2Ba5lDNI743pyzfvHln96FZ%2BmuVyKrijbeUYPDGKF8P6WAC%2Bg%3D%3D
quote(quote(OPEN_API_KEY))
# QHVhAahAshrNunAtQRenvZkOaIta0uRo9cVcSp%252Ba5lDNI743pyzfvHln96FZ%252BmuVyKrijbeUYPDGKF8P6WAC%252Bg%253D%253D
# unquote('key')


'QHVhAahAshrNunAtQRenvZkOaIta0uRo9cVcSp%252Ba5lDNI743pyzfvHln96FZ%252BmuVyKrijbeUYPDGKF8P6WAC%252Bg%253D%253D'

### 기상 일일 데이터 API 요청

In [1]:
import os
from dotenv import load_dotenv

BASE_URL = 'http://apis.data.go.kr/1360000/AsosDalyInfoService/getWthrDataList'

load_dotenv()
OPEN_API_KEY = os.getenv('OPEN_API_KEY', 'NO_KEY')

In [2]:
import requests

from datetime import datetime

class OpenAPIKeyError(Exception): ...
class OpenAPIError(Exception): ...

def fetch(start_dt, end_dt, station, page=1, size=10) -> dict:
    assert size < 1000, '최대 1,000건 까지만 호출할 수 있습니다.'

    res = requests.get(BASE_URL, params={
        'serviceKey': OPEN_API_KEY,
        'dataType': 'JSON',
        'dataCd': 'ASOS',
        'dateCd': 'DAY',

        # 필터 조건
        'startDt': start_dt,
        'endDt': end_dt,
        'stnIds': station,

        # pagination
        'numOfRows': size,
        'pageNo': page,
    })

    res.raise_for_status()

    data = res.json()

    # 1. 키 에러 처리
    if data.get('OpenAPI_ServiceResponse'):
        msg = data.get('OpenAPI_ServiceResponse')['cmmMsgHeader']['errMsg']
        raise OpenAPIKeyError(msg)

    # 2. 그 외의 에러 처리
    if data['response']['header']['resultCode'] != '00':
        msg = data['response']['header']['resultMsg']
        raise OpenAPIError(msg)

    return res.url, data

url, data = fetch('20250101', '20260825', 108, size=100)

# 오류처리 
# 1. Key가 잘못됐을 때 발생하는 오류
# {'OpenAPI_ServiceResponse': {'cmmMsgHeader': {'errMsg': 'SERVICE_KEY_IS_NULL',
#    'returnAuthMsg': '서비스 접근거부',
#    'returnReasonCode': '20'}}}

# 2. 그 외 요청 잘못됐을 때 발생 오류
# {'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL_SERVICE'},
# {'response': {'header': {'resultCode': '11', 'resultMsg': 'NO_MANDATORY

In [ ]:
# 쿼터 계산
# 540일+a 정도
# 2페이지 정도
# size = 10 일 때는 6번 정도 요청이 필요하다. 지점별로
# size = 100일때는 지점 별로 한번만 요청하면 됨.

In [ ]:
url

In [43]:
import pandas as pd

df = pd.json_normalize(data['response']['body']['items']['item'])

In [45]:
import pymysql

db_config = {
    'host':os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '3306')),
    'user': os.getenv('DB_USER', 'analysis'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'test_db'),
    'charset': 'utf8mb4'
}

def connect():
    conn = pymysql.connect(**db_config)
    return conn

In [ ]:
import re
import pandas as pd

# 전처리 (serviceKey 제거하기 -> re)
def clean_key(t):
    return re.sub(r'serviceKey=[\w\%]+&', 'serviceKey=&', t)

url = clean_key(url)
items = data['response']['body']['items']['item']
df = pd.DataFrame(items)
# df.to_dict(orient='records') 참고

In [ ]:
import json
payload = json.dumps(items[0], ensure_ascii=False)

# JSON -> str
# dict, list -> 파이썬 객체

'{"stnId": "108", "stnNm": "서울", "tm": "2025-01-01", "avgTa": "2.6", "minTa": "-2.5", "minTaHrmt": "0441", "maxTa": "8.9", "maxTaHrmt": "1540", "mi10MaxRn": "", "mi10MaxRnHrmt": "", "hr1MaxRn": "", "hr1MaxRnHrmt": "", "sumRnDur": "", "sumRn": "", "maxInsWs": "9.7", "maxInsWsWd": "290", "maxInsWsHrmt": "1417", "maxWs": "5.1", "maxWsWd": "250", "maxWsHrmt": "1519", "avgWs": "2.2", "hr24SumRws": "1861", "maxWd": "250", "avgTd": "-3.6", "minRhm": "49", "minRhmHrmt": "1543", "avgRhm": "64.3", "avgPv": "4.8", "avgPa": "1011.0", "maxPs": "1023.1", "maxPsHrmt": "2359", "minPs": "1020.0", "minPsHrmt": "1446", "avgPs": "1021.8", "ssDur": "9.6", "sumSsHr": "5.6", "hr1MaxIcsrHrmt": "1200", "hr1MaxIcsr": "1.73", "sumGsr": "8.55", "ddMefs": "", "ddMefsHrmt": "", "ddMes": "", "ddMesHrmt": "", "sumDpthFhsc": "", "avgTca": "2.6", "avgLmac": "2.6", "avgTs": "-0.2", "minTg": "-9.5", "avgCm5Te": "-0.2", "avgCm10Te": "-0.3", "avgCm20Te": "0.9", "avgCm30Te": "1.7", "avgM05Te": "3.4", "avgM10Te": "6.9", "avg

In [ ]:
import hashlib
key_str = '108|2025-01-01'
print(hashlib.sha256(key_str.encode()).hexdigest())

# len('e8f817f346d1d411cc59d5bdda64fab3763890e1f0f8f4c15805cf78874d68bf') # 64자 고정

d5bf3f4679574028419fc774dd78ba44e83dae7585b693a573bfd57106e5b781


In [52]:
SOURCE = 'ASOS_WEATHER_DAILY'
now = datetime.now()

import json
import hashlib

rows = []
for item in items:
    payload = json.dumps(item, ensure_ascii=False)
    # unique 체크를 위한 hash값
    key_str = '|'.join(map(str, [item['stnId'], item['tm']]))
    content_hash = hashlib.sha256(key_str.encode()).hexdigest()

    rows.append( (SOURCE, url, now, payload, content_hash) )

    # 배치처리
    if len(rows) == 1000:
        # db insert
        rows = []
else:
    ...

# DB INSERT 처리
conn = connect()
try:
    cur = conn.cursor()
    # INSERT IGNORE INTO raw_item(source, url, collected_at, payload, content_hash)
    #     VALUES(%s, %s, %s, %s, %s)
    INSERT_SQL = """
        INSERT INTO raw_item(source, url, collected_at, payload, content_hash)
            VALUES (%s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE
                payload = VALUES(payload),
                collected_at = NOW()
    """
    cur.executemany(INSERT_SQL, rows)
    conn.commit()
    cur.close()
    print(f'{len(rows)}건 적재 완료 !')
except Exception as e:
    print('알 수 없는 오류 발생:', e)
finally:
    conn.close()

100건 적재 완료 !
